# الدرس 04 - نمط تصميم استخدام الأدوات

في هذا الدرس ستتعلم نمط تصميم **استخدام الأدوات** لوكلاء الذكاء الاصطناعي باستخدام إطار عمل Microsoft Agent (بايثون). نغطي:

- تعريف أدوات الوظائف باستخدام المزخرف `@tool` والمعاملات ذات النوع المحدد
- توفير مخططات الأدوات حتى يعرف النموذج ما يفعله كل أداة
- التحكم في تنفيذ الأدوات باستخدام `approval_mode`
- إرجاع **مخرجات منظمة** عبر نماذج Pydantic و `response_format`

السيناريو هو **وكيل حجز رحلات** يمكنه البحث عن الوجهات، والتحقق من التوفر، واسترجاع معلومات الرحلات الجوية.


## الإعداد


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## تعريف الأدوات باستخدام المزخرف @tool

يقوم المزخرف `@tool` بتحويل وظيفة بايثون عادية إلى أداة يمكن للوكيل استدعاؤها.
النقاط الرئيسية:

- تصبح **سلسلة التوثيق** وصف الأداة الذي يراه النموذج.
- تحدد **تعليقات النوع** (بما في ذلك `Annotated` مع الوصف) مخطط الأداة.
- يتحكم `approval_mode` فيما إذا كان يجب على المستخدم الموافقة على كل استدعاء قبل تنفيذه.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## إنشاء وكيل مع عدة أدوات

مرر جميع الأدوات الثلاثة إلى العميل حتى يتمكن النموذج من استدعاء أي منها يحتاجه للإجابة على سؤال المستخدم.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## المخرجات المنظمة باستخدام الأدوات

من خلال تعيين `response_format` إلى نموذج Pydantic، يُجبر العميل على إعادة كائن JSON مُعَرَّف جيدًا بدلاً من نص حر. هذا مفيد عندما يحتاج الكود اللاحق إلى استهلاك النتيجة برمجياً.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## أنماط الموافقة على الأدوات

تتحكم المعلمة `approval_mode` في `@tool` فيما إذا كانت استدعاءات الأداة تتطلب موافقة بشرية قبل التنفيذ:

| الوضع | السلوك |
|---|---|
| `"never_require"` | تشغيل الأداة تلقائيًا — لا حاجة لتأكيد المستخدم. |
| `"always_require"` | يجب الموافقة على كل استدعاء من قبل المستخدم قبل التنفيذ. |

استخدم `"always_require"` للأدوات التي لها تأثيرات جانبية (مثل حجز رحلة طيران، خصم من بطاقة ائتمان) لكي يظل الإنسان مشتركًا في العملية.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## الملخص

في هذا الدرس تعلمت كيفية:

1. **تعريف الأدوات** باستخدام المزخرف `@tool` مع معلمات مكتوبة ووثائق تُستخدم كمخطط للأداة.
2. **تجميع أدوات متعددة** بحيث يمكن للوكيل استدعاؤها بالتسلسل للرد على الاستفسارات المعقدة.
3. **إرجاع مخرجات منظمة** من خلال تمرير نموذج Pydantic كـ `response_format`.
4. **التحكم في موافقة الأداة** باستخدام `approval_mode` للحفاظ على وجود الإنسان في العملية للعمليات الحساسة.

تشكل هذه الأنماط الأساس لبناء وكلاء موثوقين وجاهزين للإنتاج يمكنهم التفاعل مع الأنظمة الخارجية بأمان.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى لتحقيق الدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار الوثيقة الأصلية بلغتها الأصلية المصدر الأساسي والموثوق. للمعلومات الهامة، يُنصح بالاعتماد على الترجمة البشرية المهنية. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
